# Phase 1 — Cohort reporting analysis: deriving the seafood-SME topic universe

**What this does.** It reads how a cohort of large seafood companies actually report against the ESRS, and uses the *frequency* with which each Disclosure Requirement (DR) is reported to derive a defensible, cohort-evidenced **topic universe** for a downstream seafood-processing SME.

**Reproducible & local** — reads `ESRS_KPI_Mapping_V7.xlsx` directly (no Colab upload). *Run All* regenerates every table and figure.

> **Materiality caveat (read first).** This analysis measures *reporting convention* among large CSRD-scoped reporters, **not** materiality for a small SME. A frequently-reported DR is one the SME is likely to be *asked about* as the convention cascades down the value chain — a demand-relevance proxy. True materiality for any individual SME is tested separately in the validation interviews. The cohort defines the **candidate backbone**; it does not assert that every topic is material for every firm.

## 1. Config

In [ ]:
XLSX = 'ESRS_KPI_Mapping_V7.xlsx'
STACKED_SHEET = 'Mapping_ALLStacked_TU_N25'   # N=6 cohort (incl. Thai Union, 2025 reports)
MASTER_SHEET  = 'ESRS_MASTER'
TIER1_MIN = 5      # DR admitted to Tier 1 if reported by >= this many of the 6 companies
TIER2_TOPICS = ['E4','S4','E2','S3']   # sector-relevant topics rescued at topic level if below the Tier-1 cut

# cohort metadata (business model for the consistency check)
MODEL = {'Bolton':'Wild-capture (tuna)','Thai Union':'Wild-capture (tuna)','Mowi':'Aquaculture (salmon)',
         'Espersen':'Processing / brand','Nomad':'Processing / brand','Profand':'Integrated (wild+aqua)'}
GROUP = {'Bolton':'Primary (wild)','Thai Union':'Primary (wild)','Mowi':'Primary (aqua)',
         'Profand':'Primary (wild+aqua)','Espersen':'Processor/brand','Nomad':'Processor/brand'}
NAVY='#1F4E79'; GREEN='#2E8B57'; RED='#C0504D'


## 2. Load — ESRS universe + cohort mapping, build the coverage matrix

In [ ]:
import pandas as pd, numpy as np, re
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,'axes.spines.right':False})
xl = pd.ExcelFile(XLSX)

# --- ESRS master (header on the 3rd row) ---
master = pd.read_excel(xl, sheet_name=MASTER_SHEET, header=2)
master.columns = [str(c).strip() for c in master.columns]
master['DR'] = master['DR'].astype(str).str.strip()
# universe of DRs in the kept layer (drop ESRS 'Exclude' rows = governance boilerplate etc.)
keep = master[master['ESRS_LAYER'].astype(str).str.strip() != 'Exclude'].copy()
all_drs = sorted([d for d in master['DR'].unique() if re.match(r'^[ESG]\d-\d', d)])
print(f'ESRS DR universe: {len(all_drs)} disclosure requirements')

# --- cohort mapping ---
raw = pd.read_excel(xl, sheet_name=STACKED_SHEET)
raw.columns = [str(c).strip() for c in raw.columns]
for c in ['Company','Topic','DR']: raw[c] = raw[c].astype(str).str.strip()
raw['Company'] = raw['Company'].str.replace(r'\s*2025$','',regex=True)
df = raw[raw['DR'].str.match(r'^[ESG]\d-\d')].copy()
COMPANIES = sorted(df['Company'].unique())
n = len(COMPANIES)
print(f'Cohort: {n} companies -> {COMPANIES}')

# --- coverage matrix (DR x company, binary) ---
cov = pd.crosstab(df['DR'], df['Company']).reindex(columns=COMPANIES, fill_value=0)
cov[cov>1] = 1
cov['N'] = cov[COMPANIES].sum(axis=1)
cov['Topic'] = cov.index.str[:2]
print(f'DRs reported by at least one company: {len(cov)}')


## 3. Cohort overview

In [ ]:
rows=[]
for c in COMPANIES:
    rep = int(cov[c].sum())
    rows.append({'Company':c,'Business model':MODEL.get(c,'?'),'DRs reported':rep,
                 'Coverage of universe':f'{rep/len(all_drs)*100:.0f}%'})
cohort = pd.DataFrame(rows).sort_values('DRs reported',ascending=False)
print(cohort.to_string(index=False))


## 4. Reporting-frequency distribution
How many of the cohort report each DR — the core evidence.

In [ ]:
dist = cov['N'].value_counts().sort_index()
print('DRs reported by exactly k companies:')
print(dist.to_string())
fig,ax=plt.subplots(figsize=(6,3.2))
bars=ax.bar(dist.index,dist.values,color=[GREEN if k>=TIER1_MIN else '#BFBFBF' for k in dist.index],edgecolor=NAVY)
for k,v in dist.items(): ax.text(k,v+0.2,v,ha='center',weight='bold',fontsize=9)
ax.axvspan(TIER1_MIN-0.5,n+0.5,color='#E3EECF',zorder=0)
ax.set_xlabel('Number of cohort companies reporting the DR'); ax.set_ylabel('Disclosure requirements')
ax.set_title(f'Reporting frequency across the {n}-company cohort',weight='bold',fontsize=10,loc='left')
fig.savefig('p1_frequency.pdf',bbox_inches='tight'); plt.show()


## 5. Threshold sensitivity (robustness #1)
How the candidate shortlist changes with the inclusion threshold and with cohort size (N=5 vs N=6).

In [ ]:
def shortlist(cov_, comps, thr):
    sub = cov_[cov_[comps].sum(axis=1) >= thr]
    return sub
sens=[]
for thr in range(2, n+1):
    sub = cov[cov['N']>=thr]
    sens.append({'Threshold (>=k of %d)'%n:thr,'DRs':len(sub),'Topics':sub['Topic'].nunique(),
                 'Topic spread':', '.join(f"{t}:{c}" for t,c in sub['Topic'].value_counts().items())})
sens=pd.DataFrame(sens)
print('SENSITIVITY to threshold (N=%d):'%n); print(sens.to_string(index=False))

# N=5 vs N=6: recompute on the 5-company subset (drop Thai Union)
five=[c for c in COMPANIES if c!='Thai Union']
cov5=cov.copy(); cov5['N5']=cov5[five].sum(axis=1)
print('\nN=5 vs N=6, shortlist size at the (n-1) rule:')
print(f"  N=6, >=5 : {(cov['N']>=5).sum()} DRs")
print(f"  N=5, >=4 : {(cov5['N5']>=4).sum()} DRs")
common=set(cov[cov['N']>=5].index) & set(cov5[cov5['N5']>=4].index)
print(f"  overlap  : {len(common)} DRs shared by both definitions")


## 6. Business-model consistency (robustness #2)
Are the Tier-1 DRs reported across wild / aqua / processing business models, or driven by one subgroup?

In [ ]:
groups = sorted(set(GROUP.values()))
t1 = cov[cov['N']>=TIER1_MIN].copy()
def grp_reporting(dr):
    out={}
    for g in groups:
        gcomp=[c for c in COMPANIES if GROUP[c]==g]
        out[g]=int(cov.loc[dr,gcomp].sum())
    return out
cons=[]
for dr in t1.index:
    gr=grp_reporting(dr); ngrp=sum(1 for v in gr.values() if v>0)
    cons.append({'DR':dr,'Topic':dr[:2],'N':int(cov.loc[dr,'N']),**gr,
                 'Models covering':ngrp,'Cross-model':'yes' if ngrp==len(groups) else 'partial'})
cons=pd.DataFrame(cons).sort_values(['Models covering','N'],ascending=[False,False])
print('Tier-1 DRs by business-model group (groups: %s):'%groups)
print(cons.to_string(index=False))
print(f"\n{ (cons['Cross-model']=='yes').sum() } of {len(cons)} Tier-1 DRs are reported across ALL model groups (robust); the rest are concentrated.")


## 7. Inclusion rule — Tier 1 (frequency) + Tier 2 (sector rescue)
Tier 1: DR reported by >= TIER1_MIN of the cohort. Tier 2: sector-relevant topics (E4, S4, E2, S3) admitted at topic level where they fall below the Tier-1 cut but are material to seafood (validated against GRI 13 / sector literature).

In [ ]:
tier1 = sorted(cov[cov['N']>=TIER1_MIN].index)
tier2=[]
for t in TIER2_TOPICS:
    tdrs=[d for d in cov.index if d.startswith(t+'-') and d not in tier1]
    # admit the most-reported DR(s) of the rescued topic
    if tdrs:
        best=cov.loc[tdrs,'N'].sort_values(ascending=False)
        tier2 += [d for d in best.index if best[d]>=2][:3]
shortlist_drs = sorted(set(tier1)|set(tier2))
prov = {d:('Tier 1 (>=%d reporters)'%TIER1_MIN if d in tier1 else 'Tier 2 (sector rescue)') for d in shortlist_drs}
sl = pd.DataFrame({'DR':shortlist_drs,'Topic':[d[:2] for d in shortlist_drs],
                   'N':[int(cov.loc[d,'N']) for d in shortlist_drs],'Provenance':[prov[d] for d in shortlist_drs]})
print(f'Topic universe: {len(shortlist_drs)} DRs across {sl.Topic.nunique()} topics  (Tier1={len(tier1)}, Tier2={len(tier2)})')
print(sl.sort_values(['Topic','N'],ascending=[True,False]).to_string(index=False))


## 8. Expand to datapoints + ESG / quant-qual / status profile

In [ ]:
bb = keep[keep['DR'].isin(shortlist_drs)].copy()
ESG={**{f'E{i}':'Environmental' for i in range(1,6)},**{f'S{i}':'Social' for i in range(1,5)},'G1':'Governance'}
bb['ESG']=bb['Topic'].map(ESG)
print(f'Backbone datapoints (kept layer): {len(bb)}')
print('\nDatapoints per ESG pillar:'); print(bb['ESG'].value_counts().to_string())
if 'Data_Type' in bb: 
    bb['QualQuant']=np.where(bb['Data_Type'].astype(str).str.contains('narr',case=False),'Qualitative','Quantitative')
    print('\nQuant vs qual:'); print(bb['QualQuant'].value_counts().to_string())
fig,ax=plt.subplots(1,2,figsize=(9,3.4))
sl.groupby('Topic').size().reindex(['E1','E2','E3','E4','E5','S1','S2','S3','S4','G1']).fillna(0).plot.bar(ax=ax[0],color=NAVY)
ax[0].set_title('DRs per topic in the universe',fontsize=9,weight='bold'); ax[0].set_ylabel('DRs')
bb['ESG'].value_counts().plot.pie(ax=ax[1],autopct='%1.0f%%',colors=['#2E8B57','#2E5F8A','#7B3F00'])
ax[1].set_ylabel(''); ax[1].set_title('Backbone datapoints by pillar',fontsize=9,weight='bold')
fig.tight_layout(); fig.savefig('p1_universe.pdf',bbox_inches='tight'); plt.show()


## 6b. Temporal evolution — 2024 vs 2025 (robustness #3)
For the firms with both reporting years (Espersen, Nomad, Profand): is the reported set stable over time, or a moving target? A stable backbone strengthens the whole cohort approach.

In [ ]:
PAIRS={'Espersen':['Espersen_Mapping','Espersen2025_Mapping'],
       'Nomad':['Nomad_Mapping','Nomad2025_Mapping'],
       'Profand':['Profand_Mapping','Profand2025_Mapping']}
def _drs(sheet):
    d=pd.read_excel(xl,sheet_name=sheet); d.columns=[str(c).strip() for c in d.columns]
    col='DR' if 'DR' in d.columns else [c for c in d.columns if 'DR' in c][0]
    return set(x for x in d[col].astype(str).str.strip() if re.match(r'^[ESG]\d-\d',x))
eorder=list(PAIRS); eres={}
for c,(a,b) in PAIRS.items():
    A,B=_drs(a),_drs(b)
    eres[c]={'y24':len(A),'y25':len(B),'retained':sorted(A&B),'added':sorted(B-A),'dropped':sorted(A-B)}
tot24=sum(r['y24'] for r in eres.values()); ret=sum(len(r['retained']) for r in eres.values())
print('2024 -> 2025 evolution (firms with both years):')
for c,r in eres.items():
    print(f"  {c}: {r['y24']} -> {r['y25']} DRs  ({len(r['retained'])} retained, +{r['added']}, -{r['dropped']})")
print(f'\nBackbone stability: ~{ret/tot24*100:.0f}% of 2024 DRs retained in 2025.')
print('Net change is a slight contraction concentrated in advanced/forward-looking DRs')
print('(anticipated financial effects E1-9, biodiversity transition plan E4-1, non-employee S1-7,')
print(' consumer policy S4-1, action/remediation S1-4/S2-4) — consistent with the 2025 Omnibus easing.')

fig,ax=plt.subplots(1,2,figsize=(9,3.6))
for c,r in eres.items():
    ax[0].plot([0,1],[r['y24'],r['y25']],'-o',lw=2)
    ax[0].annotate(c,(1.02,r['y25']),fontsize=8,va='center')
ax[0].set_xticks([0,1]); ax[0].set_xticklabels(['2024','2025']); ax[0].set_xlim(-0.1,1.5)
ax[0].set_ylabel('ESRS DRs reported'); ax[0].set_title('A. Year-over-year reporting',fontsize=9,weight='bold',loc='left')
x=np.arange(len(eorder))
ax[1].bar(x,[len(eres[c]['retained']) for c in eorder],color='#2E8B57',label='retained')
ax[1].bar(x,[len(eres[c]['added']) for c in eorder],bottom=[len(eres[c]['retained']) for c in eorder],color='#4F81BD',label='added')
ax[1].bar(x,[-len(eres[c]['dropped']) for c in eorder],color='#C0504D',label='dropped')
ax[1].axhline(0,color='#888',lw=0.8); ax[1].set_xticks(x); ax[1].set_xticklabels(eorder,fontsize=8)
ax[1].set_title('B. DRs retained / added / dropped in 2025',fontsize=9,weight='bold',loc='left'); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig('p1_year_evolution.pdf',bbox_inches='tight'); plt.show()


## 9. Export

In [ ]:
with pd.ExcelWriter('Phase1_cohort_universe.xlsx') as w:
    cohort.to_excel(w, sheet_name='Cohort', index=False)
    cov.reset_index().to_excel(w, sheet_name='Coverage_matrix', index=False)
    sens.to_excel(w, sheet_name='Threshold_sensitivity', index=False)
    cons.to_excel(w, sheet_name='Model_consistency', index=False)
    sl.to_excel(w, sheet_name='Topic_universe', index=False)
    bb.to_excel(w, sheet_name='Backbone_datapoints', index=False)
print('exported Phase1_cohort_universe.xlsx  +  p1_frequency.pdf  +  p1_universe.pdf')


## Limitations
- **Small, purposive cohort (N=6).** Findings are illustrative, not statistically representative; the threshold sensitivity (§5) shows how the shortlist moves with the cut.
- **Reporting convention ≠ materiality.** See the caveat at the top: this is a demand-relevance proxy, validated against firms in the interviews.
- **Business-model heterogeneity.** Per-group N is small (§6); the consistency check is indicative.
- **Mixed report years / standards** (some ESRS, some GRI-mapped; some 2024, some 2025) limit strict comparability.
- **Manual mapping.** Company→DR mapping was read from reports by hand; a second-coder reliability check would strengthen it.